## INSTALL

In [ ]:
import sys

REQUIRED_VERSION = (3, 11, 9)
current_version  = sys.version_info[:3]

if current_version != REQUIRED_VERSION:
    raise RuntimeError(
        f"Wrong Python version. Expected 3.11.9, "
        f"got {'.'.join(str(v) for v in current_version)}. "
        "Make sure the correct kernel is selected."
    )

print(f"Python {'.'.join(str(v) for v in current_version)} OK")

%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
%pip install pandas numpy Pillow tqdm scikit-learn xgboost timm

## IMPORTS

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, train_test_split,
    cross_val_score, ParameterGrid
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from itertools import product

## DEFINITIONS

In [ ]:
# Load DINOv2 ViT-g/14 - Meta's self-supervised vision transformer.
# Best available frozen feature extractor for image classification (1536-d per image).
# torch.hub downloads the model weights on first run (~4.4 GB), then caches them.
dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
dino_model.eval()

# DINOv2 uses standard ImageNet normalisation at 224x224
dino_preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_and_save_images(image_dir, output_file, has_labels=True):
    # Load every image in a folder, apply the DINOv2 transform, and save to a .pt file.
    # has_labels=True  : folder has one subfolder per class
    # has_labels=False : flat folder of images
    image_tensors = []
    labels        = []
    file_paths    = []

    if has_labels:
        class_names    = sorted(os.listdir(image_dir))
        class_to_index = {name: i for i, name in enumerate(class_names)}

        # get total number of images
        total = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )

        count = 0 # tracker
        
        for class_name in class_names:
            class_folder = os.path.join(image_dir, class_name)
            
            for filename in os.listdir(class_folder):
                
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')): # if not an image file, skip
                    continue
                
                # extract image
                path = os.path.join(class_folder, filename)
                img  = Image.open(path).convert('RGB')
                image_tensors.append(dino_preprocess(img))
                labels.append(class_to_index[class_name])
                file_paths.append(path)
                
                # update progress
                count += 1
                print(f'{count}/{total}', end='\r')

        # Save tensors, labels, paths, and class mapping to .pt file
        torch.save({
            'tensors':        torch.stack(image_tensors),
            'labels':         torch.tensor(labels),
            'paths':          file_paths,
            'class_to_index': class_to_index
        }, output_file)

    else:
        
        # get list of image files
        image_files = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        
        total = len(image_files) # total number of images
        
        # extract and save each image
        for i, filename in enumerate(image_files, start=1):
            path = os.path.join(image_dir, filename)
            img  = Image.open(path).convert('RGB')
            image_tensors.append(dino_preprocess(img))
            file_paths.append(path)
            
            # update progress
            print(f'{i}/{total}', end='\r')

        # Save tensors and paths to .pt file
        torch.save({'tensors': torch.stack(image_tensors), 'paths': file_paths}, output_file)

    print(f'\nSaved {len(image_tensors)} images to {output_file}') # done!


def extract_features(input_file, output_file):
    # Pass every image through DINOv2

    # load in the images
    data   = torch.load(input_file, weights_only=False)
    images = data['tensors']
    total  = len(images)
    feature_list = []

    # pass each image through DINOv2 and save the resulting feature vector
    with torch.no_grad():
        for i, image in enumerate(images, start=1):
            # add batch dimension: (3,224,224) -> (1,3,224,224)
            feature_vector = dino_model(image.unsqueeze(0))
            feature_vector = feature_vector.squeeze().numpy()  # shape: (1536,)
            feature_list.append(feature_vector)
            print(f'{i}/{total}', end='\r')

    features = np.array(feature_list)
    
    # save features
    torch.save({**data, 'features': features}, output_file)
    print(f'\nExtracted {features.shape} -> saved to {output_file}')

In [ ]:
# --- Metrics helper -----------------------------------------------------------

def compute_metrics(true_labels, predicted_labels):
    # compute per-class balance
    classes, counts = np.unique(true_labels, return_counts=True)
    class_balance = {
        int(c): round(100 * n / len(true_labels), 1)
        for c, n in zip(classes, counts)
    }
    
    return {
        'accuracy':      accuracy_score(true_labels, predicted_labels),
        'f1':            f1_score(true_labels, predicted_labels, average='weighted'),
        'precision':     precision_score(true_labels, predicted_labels, average='weighted'),
        'recall':        recall_score(true_labels, predicted_labels, average='weighted'),
        'class_balance': class_balance,
    }


def cv_report(model_name, fold_results):
    # print mean and the std across all folds for each metric
    print(f'--- {model_name} ---')
    for metric in ('accuracy', 'f1', 'precision', 'recall'):
        values = [r[metric] for r in fold_results]
        print(f'  {metric:<9}: {np.mean(values):.4f} +/- {np.std(values):.4f}')
        
    # print the balance for analysis later
    print('  class balance:')
    for class_id, pct in fold_results[0]['class_balance'].items():
        print(f'    class {class_id}: {pct}%')
    print()


# --- MLP model ----------------------------------------------------------------
class MLP(nn.Module):
    # A stack of fully-connected layers with activation and optional dropout
    def __init__(self, input_size, num_classes, hidden_layers=(256, 128),
                 activation=nn.ReLU, dropout=0.0):
        super().__init__()
        
        # variables of the MLP object
        layers       = []
        current_size = input_size
        
        # build the network given input parameters
        for layer_size in hidden_layers:
            layers.append(nn.Linear(current_size, layer_size))
            layers.append(activation())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            current_size = layer_size
            
        # final layer to output class classifier logits
        layers.append(nn.Linear(current_size, num_classes))
        self.network = nn.Sequential(*layers)

    # forward pass through the network
    def forward(self, x):
        return self.network(x)


# --- Classical Classifier functions ------------------------------------------------
# All functions take numpy arrays and return (trained_model, metrics_dict).

def train_logistic_regression(train_features, train_labels, val_features, val_labels, C=1.0):
    
    model = LogisticRegression(C=C, max_iter=2000)
    model.fit(np.asarray(train_features, np.float64), np.asarray(train_labels, np.int64))
    preds = model.predict(np.asarray(val_features, np.float64))
    
    return model, compute_metrics(np.asarray(val_labels, np.int64), preds)


def train_svm(train_features, train_labels, val_features, val_labels,
              kernel='rbf', C=1.0, gamma='scale'):
    
    model = SVC(kernel=kernel, C=C, gamma=gamma, decision_function_shape='ovr')
    model.fit(np.asarray(train_features, np.float64), np.asarray(train_labels, np.int64))
    preds = model.predict(np.asarray(val_features, np.float64))
    
    return model, compute_metrics(np.asarray(val_labels, np.int64), preds)


def train_random_forest(train_features, train_labels, val_features, val_labels, n_trees=300):
    model = RandomForestClassifier(n_estimators=n_trees, max_features='sqrt',
                                   random_state=42, n_jobs=-1)
    
    model.fit(np.asarray(train_features, np.float64), np.asarray(train_labels, np.int64))
    preds = model.predict(np.asarray(val_features, np.float64))
    
    return model, compute_metrics(np.asarray(val_labels, np.int64), preds)


def train_xgboost(train_features, train_labels, val_features, val_labels):
    
    # we use softmax objective for multi-class classification, and logloss as the evaluation metric
    model = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0)
    model.fit(np.asarray(train_features, np.float64), np.asarray(train_labels, np.int64))
    preds = model.predict(np.asarray(val_features, np.float64))
    
    return model, compute_metrics(np.asarray(val_labels, np.int64), preds)


def train_mlp(train_features, train_labels, val_features, val_labels,
              hidden_layers=(256, 128), activation=nn.ReLU,
              dropout=0.0, epochs=200, learning_rate=1e-3):
    
    device      = 'cuda' if torch.cuda.is_available() else 'cpu' # much faster with GPU, but still works on CPU
    num_classes = len(np.unique(train_labels))

    # move all the data to the device 
    X_train = torch.tensor(np.asarray(train_features, np.float32)).to(device)
    y_train = torch.tensor(np.asarray(train_labels,   np.int64)).to(device)
    X_val   = torch.tensor(np.asarray(val_features,   np.float32)).to(device)
    y_val   = torch.tensor(np.asarray(val_labels,     np.int64)).to(device)

    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True) # 256 is default

    # create the MLP model, optimizer, and loss function
    model     = MLP(X_train.shape[1], num_classes, hidden_layers, activation, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_fn   = nn.CrossEntropyLoss()

    # early stopping, if no improvement occurs we revert to best weights
    best_val_loss    = float('inf')
    best_weights     = None
    no_improve_count = 0

    # training loop
    for epoch in range(epochs):
        model.train() # set model to training mode
        
        # train for one epoch
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval() # set model to evaluation mode for validation
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item() # compute validation loss

        # early stopping check
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_weights     = model.state_dict()
            no_improve_count = 0
        else:
            no_improve_count += 1
            if no_improve_count >= 25:
                break

                
    model.load_state_dict(best_weights) # revert to best weights
    model.eval() # basically turn off dropout for evaluation
    
    with torch.no_grad():
        predictions = model(X_val).argmax(dim=1).cpu().numpy() # get predicted class

    return model, compute_metrics(np.asarray(val_labels, np.int64), predictions)

In [ ]:
def fine_tune_dino(images_file, train_idx, val_idx, num_classes,
                   num_blocks_to_unfreeze=4,
                   backbone_lr=1e-5, head_lr=1e-3,
                   dropout=0.1, epochs=20, batch_size=16):
    
    # NOTE: GPU strongly recommended. On CPU each epoch takes several minutes.
    
    # select device and load the data
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}  (GPU strongly recommended for fine-tuning)')

    data   = torch.load(images_file, weights_only=False)
    images = data['tensors']   # (N, 3, 224, 224)
    labels = data['labels']    # (N,)

    # DINOv2 ViT-g/14 backbone block: a layer that does self-attention
        # Frozen during fine-tuning because the pretrained
        # weights already encode strong visual features so retraining them risks
        # destroying that knowledge while only the task head needs to learn.
        
    # So we:
    # --- Freeze entire backbone, then selectively unfreeze ---
    
    for param in dino_model.parameters(): # freeze all parameters in the backbone
        param.requires_grad = False

    num_total_blocks = len(dino_model.blocks) 
    
    for block in dino_model.blocks[num_total_blocks - num_blocks_to_unfreeze:]: # unfreeze the last few blocks and the final norm layer
        for param in block.parameters():
            param.requires_grad = True
    for param in dino_model.norm.parameters(): # unfreeze final norm layer
        param.requires_grad = True

    trainable = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in dino_model.parameters())
    
    print(f'Trainable backbone params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

    # --- Classification head ---
    head = nn.Sequential(
        nn.Linear(1536, 512),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    ).to(device)

    dino_model.to(device) # move to device

    # we select the AdamW optimizer which is a good default for fine-tuning large pretrained models, with separate learning rates for backbone and head
    optimizer = torch.optim.AdamW([
        {'params': [p for p in dino_model.parameters() if p.requires_grad], 'lr': backbone_lr},
        {'params': head.parameters(), 'lr': head_lr},
    ], weight_decay=1e-4)

    # we use cross-entropy loss for classification
    # cross-entropy loss is: a standard loss function for multi-class classification tasks that measures the difference between predicted probabilities and true class labels, encouraging the model to assign high probabilities to the correct classes.
    loss_fn      = nn.CrossEntropyLoss()
    train_loader = DataLoader(
        TensorDataset(images[train_idx], labels[train_idx]),
        batch_size=batch_size, shuffle=True
    )

    # We only save the weights that actually changed
    unfrozen_prefixes = tuple(
        f'blocks.{i}.'
        for i in range(num_total_blocks - num_blocks_to_unfreeze, num_total_blocks)
    ) + ('norm.',)

    # early stopping variables
    best_val_loss       = float('inf')
    best_backbone_state = None
    best_head_state     = None
    no_improve          = 0

    # retraining loop
    for epoch in range(epochs):
        dino_model.train()
        head.train()
        
        # train for one epoch
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(head(dino_model(X_batch)), y_batch)
            loss.backward()
            optimizer.step()

        # validation step
        dino_model.eval()
        head.eval()
        
        with torch.no_grad():
            val_images = images[val_idx]
            val_feats  = []
            
            for i in range(0, len(val_images), batch_size):
                val_feats.append(dino_model(val_images[i:i+batch_size].to(device)).cpu())
                
            val_feats = torch.cat(val_feats).to(device)
            val_loss  = loss_fn(head(val_feats), labels[val_idx].to(device)).item()

        print(f'Epoch {epoch+1:>3}/{epochs}  val_loss={val_loss:.4f}', end='\r')

        # early stopping check
        if val_loss < best_val_loss:
            best_val_loss       = val_loss
            best_backbone_state = {k: v.cpu().clone() for k, v in dino_model.state_dict().items()
                                   if k.startswith(unfrozen_prefixes)}
            best_head_state     = {k: v.cpu().clone() for k, v in head.state_dict().items()}
            no_improve          = 0
        else:
            no_improve += 1
            if no_improve >= 7:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break

    print(f'\nBest val_loss: {best_val_loss:.4f}')

    # Restore the best weights found during training
    state = dino_model.state_dict()
    state.update(best_backbone_state)
    dino_model.load_state_dict(state)
    head.load_state_dict(best_head_state)

    # Final val predictions
    dino_model.eval()
    head.eval()
    with torch.no_grad():
        val_feats = []
        for i in range(0, len(val_images), batch_size):
            val_feats.append(dino_model(val_images[i:i+batch_size].to(device)).cpu())
        val_preds = head(torch.cat(val_feats).to(device)).argmax(dim=1).cpu().numpy()

    # Move model back to CPU and re-freeze everything for use by extract_features
    dino_model.cpu()
    for param in dino_model.parameters():
        param.requires_grad = False

    return head, compute_metrics(labels[val_idx].numpy(), val_preds)

## PREPROCESSING
Load images from disk, apply the DINOv2 transform, and save as tensors.

In [ ]:
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'

if not os.path.exists(train_image_dir):
    raise FileNotFoundError(f'Training folder not found: {train_image_dir}')
if not os.path.exists(test_image_dir):
    raise FileNotFoundError(f'Test folder not found: {test_image_dir}')

# Save preprocessed tensors to disk (only needs to run once)
load_and_save_images(train_image_dir, r'Data\task1_data\t1_train.pt', has_labels=True)
load_and_save_images(test_image_dir,  r'Data\task1_data\t1_test.pt',  has_labels=False)

## FEATURE EXTRACTION
Run every image through DINOv2 ViT-g/14 and save the 1536-d feature vectors.

In [ ]:
# Run images through DINOv2 and save 1536-d feature vectors (only needs to run once)
extract_features(r'Data\task1_data\t1_train.pt', r'Data\task1_data\t1_train_features.pt')
extract_features(r'Data\task1_data\t1_test.pt',  r'Data\task1_data\t1_test_features.pt')

## FINE-TUNING
Optionally fine-tune the last few DINOv2 blocks on your data, then re-extract features.
Skip this section if you want to use frozen features only.

In [ ]:
# Fine-tune the last 4 DINOv2 blocks on task 1 training images.
# After this runs, re-extract features from the adapted backbone so the rest of
# the notebook (LOAD DATA, EVAL, HYPER SEARCH) automatically benefits.
#
# GPU strongly recommended - on CPU this will take a long time.

num_classes  = len(np.unique(torch.load(r'Data\task1_data\t1_train.pt',
                                        weights_only=False)['labels'].numpy()))
all_idx      = np.arange(len(torch.load(r'Data\task1_data\t1_train.pt',
                                        weights_only=False)['labels']))
train_idx_ft, val_idx_ft = train_test_split(
    all_idx, test_size=0.2, random_state=42,
    stratify=torch.load(r'Data\task1_data\t1_train.pt',
                        weights_only=False)['labels'].numpy()
)

fine_tuned_head, ft_metrics = fine_tune_dino(
    images_file            = r'Data\task1_data\t1_train.pt',
    train_idx              = train_idx_ft,
    val_idx                = val_idx_ft,
    num_classes            = num_classes,
    num_blocks_to_unfreeze = 4,
    backbone_lr            = 1e-5,
    head_lr                = 1e-3,
    dropout                = 0.1,
    epochs                 = 20,
    batch_size             = 16,
)
print(f'Fine-tuned val:  F1={ft_metrics["f1"]:.4f}  Acc={ft_metrics["accuracy"]:.4f}')

# Re-extract features using the now-adapted backbone and save to new files.
# In the LOAD DATA cell, set USE_FINETUNED = True to use these.
print('\nRe-extracting features with fine-tuned backbone...')
extract_features(r'Data\task1_data\t1_train.pt', r'Data\task1_data\t1_train_features_ft.pt')
extract_features(r'Data\task1_data\t1_test.pt',  r'Data\task1_data\t1_test_features_ft.pt')
print('Done. Set USE_FINETUNED = True in the LOAD DATA cell to use these features.')

## LOAD DATA

In [ ]:
# Set USE_FINETUNED = True after running the FINE-TUNING cell to use the adapted features
USE_FINETUNED = False

if USE_FINETUNED:
    train_data = torch.load(r'Data\task1_data\t1_train_features_ft.pt', weights_only=False)
    test_data  = torch.load(r'Data\task1_data\t1_test_features_ft.pt',  weights_only=False)
else:
    train_data = torch.load(r'Data\task1_data\t1_train_features.pt', weights_only=False)
    test_data  = torch.load(r'Data\task1_data\t1_test_features.pt',  weights_only=False)

train_features = train_data['features']        # shape: (3750, 1536)
train_labels   = train_data['labels'].numpy()  # shape: (3750,)
test_features  = test_data['features']         # shape: (1250, 1536) - no labels, used for submission

print(f'Train: {train_features.shape}  Labels: {train_labels.shape}')
print(f'Test (submission): {test_features.shape}')
print(f'Classes: {train_data["class_to_index"]}')

# Split labelled data into train and validation sets.
# stratify=train_labels keeps the same class proportions in both splits.
train_feat, val_feat, train_lbl, val_lbl = train_test_split(
    train_features, train_labels,
    test_size=0.2, random_state=42, stratify=train_labels
)
print(f'\nTrain split: {train_feat.shape}  Val split: {val_feat.shape}')

## MODEL EVALUATION

In [ ]:
# --- Settings ----------------------------------------------------------------
SVM_KERNEL = 'rbf'    # 'linear' | 'rbf' | 'poly'
# -----------------------------------------------------------------------------
# Quick 5-fold CV for the fast models.

cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Running 5-fold cross validation...\n')

# Logistic Regression
fold_results = []
for train_idx, val_idx in cross_val.split(train_features, train_labels):
    _, metrics = train_logistic_regression(
        train_features[train_idx], train_labels[train_idx],
        train_features[val_idx],   train_labels[val_idx]
    )
    fold_results.append(metrics)
cv_report('Logistic Regression (5-fold CV)', fold_results)

# SVM
fold_results = []
for train_idx, val_idx in cross_val.split(train_features, train_labels):
    _, metrics = train_svm(
        train_features[train_idx], train_labels[train_idx],
        train_features[val_idx],   train_labels[val_idx],
        kernel=SVM_KERNEL
    )
    fold_results.append(metrics)
cv_report(f'SVM kernel={SVM_KERNEL} (5-fold CV)', fold_results)

# XGBoost
fold_results = []
for train_idx, val_idx in cross_val.split(train_features, train_labels):
    _, metrics = train_xgboost(
        train_features[train_idx], train_labels[train_idx],
        train_features[val_idx],   train_labels[val_idx]
    )
    fold_results.append(metrics)
cv_report('XGBoost (5-fold CV)', fold_results)

## HYPER PARAM SEARCH

In [ ]:
# --- SVM grid search ---------------------------------------------------------
# We try every combination of kernel and its relevant hyperparameters.
# C      : how strictly to separate classes (higher = less margin, more precise)
# gamma  : how far each training point's influence reaches (rbf/poly/sigmoid only)
# degree : polynomial degree (poly only)
svm_param_grid = [
    {'kernel': ['linear'],  'C': [0.01, 0.1, 1, 10, 100]},
    {'kernel': ['rbf'],     'C': [0.1, 1, 10, 100],  'gamma': ['scale', 'auto', 0.01, 0.001]},
    {'kernel': ['poly'],    'C': [0.1, 1, 10],        'gamma': ['scale', 'auto'], 'degree': [2, 3, 4]},
]
svm_configs = list(ParameterGrid(svm_param_grid))

print(f'SVM search: {len(svm_configs)} configs, 5-fold CV each')
best_svm_f1     = -1
best_svm_params = {}

for params in tqdm.tqdm(svm_configs, desc='SVM'):
    model  = SVC(decision_function_shape='ovr', **params)
    scores = cross_val_score(model, train_feat, train_lbl, cv=5, scoring='f1_weighted', n_jobs=-1)
    if scores.mean() > best_svm_f1:
        best_svm_f1     = scores.mean()
        best_svm_params = params

print(f'\nBest SVM: {best_svm_params}  F1={best_svm_f1:.4f}')

# --- MLP grid search ---------------------------------------------------------
mlp_configs = list(product(
    [(512, 256), (1024, 512), (1024, 512, 256), (512, 256, 128, 64)],
    [nn.ReLU, nn.GELU],
    [0.0, 0.2],
    [1e-3, 3e-4],
))

print(f'\nMLP search: {len(mlp_configs)} configs')
best_mlp_f1     = -1
best_mlp_params = {}

for hidden, activation, dropout, lr in tqdm.tqdm(mlp_configs, desc='MLP'):
    _, metrics = train_mlp(train_feat, train_lbl, val_feat, val_lbl,
                           hidden_layers=hidden, activation=activation,
                           dropout=dropout, learning_rate=lr)
    if metrics['f1'] > best_mlp_f1:
        best_mlp_f1     = metrics['f1']
        best_mlp_params = dict(hidden_layers=hidden, activation=activation.__name__,
                               dropout=dropout, learning_rate=lr)

print(f'\nBest MLP: {best_mlp_params}  F1={best_mlp_f1:.4f}')

## SUBMISSION

In [ ]:
# --- Settings ----------------------------------------------------------------
# Choose a model and paste in the best params from the search above
SUBMISSION_MODEL = 'linear'   # 'linear' | 'svm' | 'xgboost' | 'mlp'

SVM_SUBMIT_PARAMS = {'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}

MLP_SUBMIT_HIDDEN   = (512, 256)
MLP_SUBMIT_DROPOUT  = 0.2
MLP_SUBMIT_EPOCHS   = 200
MLP_SUBMIT_LR       = 3e-4
# -----------------------------------------------------------------------------

all_features    = np.asarray(train_features, np.float64)
all_labels      = np.asarray(train_labels,   np.int64)
submit_features = np.asarray(test_features,  np.float64)

print(f'Training {SUBMISSION_MODEL} on all {len(all_features)} labelled samples...')

if SUBMISSION_MODEL == 'linear':
    model = LogisticRegression(max_iter=2000)
    model.fit(all_features, all_labels)
    predictions = model.predict(submit_features)

elif SUBMISSION_MODEL == 'svm':
    model = SVC(decision_function_shape='ovr', **SVM_SUBMIT_PARAMS)
    model.fit(all_features, all_labels)
    predictions = model.predict(submit_features)

elif SUBMISSION_MODEL == 'xgboost':
    model = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0)
    model.fit(all_features, all_labels)
    predictions = model.predict(submit_features)

elif SUBMISSION_MODEL == 'mlp':
    # MLP needs a small validation set for early stopping, so we hold out 5%
    sub_train_feat, sub_val_feat, sub_train_lbl, sub_val_lbl = train_test_split(
        train_features, train_labels, test_size=0.05, random_state=42, stratify=train_labels
    )
    model, _ = train_mlp(
        sub_train_feat, sub_train_lbl, sub_val_feat, sub_val_lbl,
        hidden_layers=MLP_SUBMIT_HIDDEN, dropout=MLP_SUBMIT_DROPOUT,
        epochs=MLP_SUBMIT_EPOCHS, learning_rate=MLP_SUBMIT_LR
    )
    model.eval()
    with torch.no_grad():
        device        = next(model.parameters()).device
        submit_tensor = torch.tensor(np.asarray(test_features, np.float32)).to(device)
        predictions   = model(submit_tensor).argmax(dim=1).cpu().numpy()

else:
    raise ValueError(f'Unknown model: {SUBMISSION_MODEL}')

image_ids  = [os.path.splitext(os.path.basename(p))[0] for p in test_data['paths']]
submission = pd.DataFrame({'image_id': image_ids, 'class_id': predictions})
submission.to_csv('t1_submission.csv', index=False)
print(f'Saved {len(submission)} predictions to t1_submission.csv')